## Why Pure Vector Search Fails in Production
If you rely solely on dense vector embeddings (semantic search) for RAG, your system will often fail in two common scenarios:

**Exact Keyword Lookups:** If a user searches for a specific error code (ERR_CONNECTION_RESET), product SKU (XYZ-9988), or technical term, vector search models frequently blur semantic meanings and miss exact string matches.

**Keyword Over-indexing:** Conversely, traditional keyword search (BM25 / TF-IDF) misses semantic intent (e.g., searching for "database timeout" won't match a document that says "SQL connection latency dropped").

In [2]:
from rank_bm25 import BM25Okapi

In [3]:
# 1. Sample Corpus of Technical Documents
corpus = [
    (
        "doc_1",
        "FastAPI is a modern, high-performance web framework for building APIs"
        " with Python.",
    ),
    (
        "doc_2",
        "LangGraph allows building stateful, multi-agent actor applications with"
        " graph topologies.",
    ),
    (
        "doc_3",
        "Error code ERR_CONNECTION_RESET occurs when the TCP connection is"
        " abruptly dropped.",
    ),
    (
        "doc_4",
        "PostgreSQL provides robust long-term user persistence and relational"
        " schema design.",
    ),
]

In [9]:
# 2. setup BM25 Sparse Index

tokenized_corpus = [doc[1].lower().split(" ") for doc in corpus]
tokenized_corpus



[['fastapi',
  'is',
  'a',
  'modern,',
  'high-performance',
  'web',
  'framework',
  'for',
  'building',
  'apis',
  'with',
  'python.'],
 ['langgraph',
  'allows',
  'building',
  'stateful,',
  'multi-agent',
  'actor',
  'applications',
  'with',
  'graph',
  'topologies.'],
 ['error',
  'code',
  'err_connection_reset',
  'occurs',
  'when',
  'the',
  'tcp',
  'connection',
  'is',
  'abruptly',
  'dropped.'],
 ['postgresql',
  'provides',
  'robust',
  'long-term',
  'user',
  'persistence',
  'and',
  'relational',
  'schema',
  'design.']]

In [10]:
bm25 = BM25Okapi(tokenized_corpus)
bm25

In [14]:
# 3. Simulate Hybrid Retrieval

query = "How do I fix ERR_CONNECTION_RESET"
tokenized_query = query.lower().split(" ")

In [15]:
# Get BM25 keyword scores

bm25_scores = bm25.get_scores(tokenized_query)
print("--- BM25 Keyword Search Scores ---")
for doc, score in zip(corpus, bm25_scores):
  print(f"[{score:.2f}] {doc[0]}: {doc[1]}")

--- BM25 Keyword Search Scores ---
[0.00] doc_1: FastAPI is a modern, high-performance web framework for building APIs with Python.
[0.00] doc_2: LangGraph allows building stateful, multi-agent actor applications with graph topologies.
[0.84] doc_3: Error code ERR_CONNECTION_RESET occurs when the TCP connection is abruptly dropped.
[0.00] doc_4: PostgreSQL provides robust long-term user persistence and relational schema design.


In [13]:
# 4. Re-ranking Simulation (In production, you would call Cohere Rerank API)
# client.rerank(query=query, documents=[d[1] for d in corpus], top_n=2)
print("\n--- Re-ranking Pipeline (Top Results Selected) ---")
print("Selected: doc_3 (Exact keyword & semantic match for error code)")


--- Re-ranking Pipeline (Top Results Selected) ---
Selected: doc_3 (Exact keyword & semantic match for error code)


In [17]:
import string
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

# ---------------------------------------------------------
# 1. Production Tokenization & Preprocessing (Fixes Punctuation)
# ---------------------------------------------------------


def clean_and_tokenize(text: str) -> list[str]:
  """Tokenizes text by lowercasing and stripping punctuation

  to ensure exact error code and keyword matching.
  """
  text = text.lower()
  # Remove punctuation that breaks exact string/code matching (e.g. underscores, hyphens)
  translator = str.maketrans("", "", string.punctuation)
  cleaned_text = text.translate(translator)
  return cleaned_text.split()


# ---------------------------------------------------------
# 2. Production Hybrid Retrieval Engine Class
# ---------------------------------------------------------


class ProductionHybridRetriever:

  def __init__(self, documents: list[dict]):
    """Documents format: [{"id": "doc_1", "text": "..."}]"""
    self.documents = documents
    self.corpus_texts = [doc["text"] for doc in documents]

    # Initialize Tokenized BM25 Index
    tokenized_corpus = [clean_and_tokenize(text) for text in self.corpus_texts]
    self.bm25 = BM25Okapi(tokenized_corpus)

    # Initialize Cross-Encoder for Re-ranking (Loaded locally or via API)
    # cross-encoder/ms-marco-MiniLM-L-6-v2 is standard for high-speed local re-ranking
    print("Loading Cross-Encoder re-ranker model...")
    self.reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

  def sparse_search(self, query: str, top_k: int = 3) -> list[dict]:
    """Executes BM25 sparse keyword search."""
    tokenized_query = clean_and_tokenize(query)
    scores = self.bm25.get_scores(tokenized_query)

    # Pair scores with documents and sort descending
    scored_docs = sorted(
        zip(self.documents, scores), key=lambda x: x[1], reverse=True
    )
    return [
        {"doc": doc, "score": float(score), "source": "bm25"}
        for doc, score in scored_docs[:top_k]
    ]

  def hybrid_retrieve_and_rerank(
      self, query: str, top_candidates: int = 5, final_top_k: int = 2
  ) -> list[dict]:
    """Combines candidate retrieval with a true Cross-Encoder re-ranking step."""
    # Step 1: Retrieve candidate pool (In production, combine sparse BM25 + dense Vector search here)
    candidates = self.sparse_search(query, top_k=top_candidates)
    candidate_texts = [item["doc"]["text"] for item in candidates]

    if not candidate_texts:
      return []

    # Step 2: Cross-Encoder Re-ranking
    # Cross-encoders process [query, document] simultaneously to evaluate deep semantic relevance
    pairs = [[query, text] for text in candidate_texts]
    rerank_scores = self.reranker.predict(pairs)

    # Step 3: Sort by Cross-Encoder score
    reranked_results = []
    for item, score in zip(candidates, rerank_scores):
      reranked_results.append({
          "id": item["doc"]["id"],
          "text": item["doc"]["text"],
          "relevance_score": float(score),
      })

    reranked_results = sorted(
        reranked_results, key=lambda x: x["relevance_score"], reverse=True
    )
    return reranked_results[:final_top_k]


# ---------------------------------------------------------
# Test Execution
# ---------------------------------------------------------
if __name__ == "__main__":
  corpus = [
      {
          "id": "doc_1",
          "text": (
              "FastAPI is a modern, high-performance web framework for"
              " building APIs with Python."
          ),
      },
      {
          "id": "doc_2",
          "text": (
              "LangGraph allows building stateful, multi-agent actor"
              " applications with graph topologies."
          ),
      },
      {
          "id": "doc_3",
          "text": (
              "Error code ERR_CONNECTION_RESET occurs when the TCP connection"
              " is abruptly dropped by the server."
          ),
      },
      {
          "id": "doc_4",
          "text": (
              "PostgreSQL provides robust long-term user persistence and"
              " relational schema design."
          ),
      },
  ]

  retriever = ProductionHybridRetriever(corpus)

  user_query = "How do I fix ERR_CONNECTION_RESET?"
  results = retriever.hybrid_retrieve_and_rerank(
      user_query, top_candidates=3, final_top_k=2
  )

  print(f"\n--- Final Re-ranked Results for Query: '{user_query}' ---")
  for res in results:
    print(
        f"[{res['id']}] Score: {res['relevance_score']:.4f} | Text:"
        f" {res['text']}"
    )

Loading Cross-Encoder re-ranker model...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3462.28it/s]



--- Final Re-ranked Results for Query: 'How do I fix ERR_CONNECTION_RESET?' ---
[doc_3] Score: 4.9621 | Text: Error code ERR_CONNECTION_RESET occurs when the TCP connection is abruptly dropped by the server.
[doc_1] Score: -11.4349 | Text: FastAPI is a modern, high-performance web framework for building APIs with Python.


## Production-Grade Hybrid Search & Re-ranking Implementation

In [ ]:
import re #Tokenization: handles words and technical identifiers -- regular expressions
import numpy as np

from rank_bm25 import BM25Okapi #Sparse retrieval based on keyword relevance

from sentence_transformers import (
    SentenceTransformer, #Dense retrieval using semantic embeddings
    CrossEncoder, #Reranks retrieved documents based on query relevance
)

In [3]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7502.05it/s]


In [7]:

corpus = [
    {
        "id": "doc_1",
        "text": (
            "FastAPI is a modern, high-performance web framework for"
            " building APIs with Python."
        ),
    },
    {
        "id": "doc_2",
        "text": (
            "LangGraph allows building stateful, multi-agent actor"
            " applications with graph topologies."
        ),
    },
    {
        "id": "doc_3",
        "text": (
            "Error code ERR_CONNECTION_RESET occurs when the TCP connection"
            " is abruptly dropped by the server."
        ),
    },
    {
        "id": "doc_4",
        "text": (
            "PostgreSQL provides robust long-term user persistence and"
            " relational schema design."
        ),
    },
]

In [8]:
# Preserve underscores and hyphens in technical identifiers


def clean_and_tokenize(text: str) -> list[str]:
    """Lowercase text and tokenize while preserving technical identifiers."""
    return re.findall(
        r"[a-z0-9_]+(?:-[a-z0-9_]+)*",
        text.lower()
    )

In [ ]:
corpus_texts = [doc["text"] for doc in corpus]

tokenized_corpus = [clean_and_tokenize(text) for text in corpus_texts]

tokenized_corpus

In [ ]:
# builds the BM25 sparse index.
bm25 = BM25Okapi(tokenized_corpus)
bm25 

In [ ]:
query = "How do I fix ERR_CONNECTION_RESET?"
# “How relevant is each document to this query based on keyword matching?”
tokenized_query = clean_and_tokenize(query)

bm25_scores = bm25.get_scores(tokenized_query)

for doc, score in zip(corpus, bm25_scores):
    print(
        f"[{score:.4f}] {doc['id']} | {doc['text']}"
    )

[0.0000] doc_1 | FastAPI is a modern, high-performance web framework for building APIs with Python.
[0.0000] doc_2 | LangGraph allows building stateful, multi-agent actor applications with graph topologies.
[0.7718] doc_3 | Error code ERR_CONNECTION_RESET occurs when the TCP connection is abruptly dropped by the server.
[0.0000] doc_4 | PostgreSQL provides robust long-term user persistence and relational schema design.


In [13]:
scored_docs = sorted(
    zip(corpus, bm25_scores),
    key=lambda x: x[1],
    reverse=True,
)

top_candidates = scored_docs[:3]

for doc, score in top_candidates:
    print(
        f"[{score:.4f}] {doc['id']} | {doc['text']}"
    )

[0.7718] doc_3 | Error code ERR_CONNECTION_RESET occurs when the TCP connection is abruptly dropped by the server.
[0.0000] doc_1 | FastAPI is a modern, high-performance web framework for building APIs with Python.
[0.0000] doc_2 | LangGraph allows building stateful, multi-agent actor applications with graph topologies.


### BM25 Retrieval Flow

**Query → Tokenize → BM25 Scores → Sort by Score → Top 3 Candidates**



Next step: dense retrieval setup.

In [14]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

doc_embeddings = embedding_model.encode(
    corpus_texts,
    normalize_embeddings=True
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7364.57it/s]


In [15]:
doc_embeddings

array([[-0.08993303,  0.01838201, -0.06867057, ...,  0.07824386,
         0.10158712,  0.10134111],
       [-0.01491481, -0.04352282, -0.04229178, ..., -0.02528696,
        -0.03384642, -0.00992699],
       [-0.04780663, -0.04003478,  0.01593442, ...,  0.10441694,
        -0.06669482, -0.06219973],
       [-0.04068729, -0.00409853, -0.04082488, ..., -0.00718028,
         0.01324104,  0.0982453 ]], shape=(4, 384), dtype=float32)

In [17]:
# convert the query into an embedding.

query_embedding = embedding_model.encode(
    query,
    normalize_embeddings=True
)
query_embedding

array([-3.80691364e-02, -5.02726585e-02, -2.24739593e-02,  2.07984280e-02,
       -7.10228905e-02, -1.64169818e-02,  2.38074958e-02,  3.28618404e-03,
        2.02473644e-02, -3.74178030e-02,  1.68421101e-02,  5.56634888e-02,
       -2.98059806e-02,  4.99459989e-02, -5.41061834e-02,  3.38643678e-02,
       -6.35267571e-02,  4.58284840e-02,  5.42158596e-02, -7.08744570e-04,
       -1.16290018e-01,  6.24493249e-02, -9.12404209e-02,  3.43718156e-02,
        5.86502254e-02, -9.36734527e-02,  2.21750196e-02,  2.65826788e-02,
       -6.43122867e-02, -1.97315458e-02, -8.74151301e-04, -1.45994890e-02,
       -1.06654018e-01, -4.17154655e-02,  4.79391292e-02,  8.32647607e-02,
       -1.01405056e-02,  2.68390235e-02, -8.68061744e-03,  1.44785820e-02,
        4.67521399e-02, -4.51282412e-02, -5.06675541e-02, -2.42735706e-02,
        1.42248217e-02,  5.93331382e-02, -8.12990067e-04, -1.11821452e-02,
        5.08876927e-02, -2.26983279e-02,  7.37401694e-02,  1.04975253e-01,
       -4.98958305e-02, -

**Query Embedding → Compare with Document Embeddings → Similarity Scores**

In [19]:
dense_scores = doc_embeddings @ query_embedding

dense_scores

array([-0.03432666, -0.05775398,  0.7496427 ,  0.06141432], dtype=float32)

In [20]:
# sort the dense similarity scores and select the top candidates.

dense_ranked = sorted(
    zip(corpus, dense_scores),
    key=lambda x: x[1],
    reverse=True,
)

top_dense_candidates = dense_ranked[:3]

In [21]:
for doc, score in top_dense_candidates:
    print(
        f"[{score:.4f}] {doc['id']} | {doc['text']}"
    )

[0.7496] doc_3 | Error code ERR_CONNECTION_RESET occurs when the TCP connection is abruptly dropped by the server.
[0.0614] doc_4 | PostgreSQL provides robust long-term user persistence and relational schema design.
[-0.0343] doc_1 | FastAPI is a modern, high-performance web framework for building APIs with Python.


In [22]:
def reciprocal_rank_fusion(rankings, k: int = 60):
    rrf_scores = {}

    for ranking in rankings:
        for rank, (doc, _) in enumerate(ranking, start=1):
            doc_id = doc["id"]

            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + (
                1 / (k + rank)
            )

    return rrf_scores

In [23]:
rrf_scores = reciprocal_rank_fusion([
    top_candidates,
    top_dense_candidates,
])

In [25]:
hybrid_ranked = sorted(
    rrf_scores.items(),
    key=lambda x: x[1],
    reverse=True,
)

hybrid_ranked

[('doc_3', 0.03278688524590164),
 ('doc_1', 0.03200204813108039),
 ('doc_4', 0.016129032258064516),
 ('doc_2', 0.015873015873015872)]

In [26]:
doc_lookup = {
    doc["id"]: doc
    for doc in corpus
}

hybrid_candidates = [
    {
        "doc": doc_lookup[doc_id],
        "rrf_score": score,
    }
    for doc_id, score in hybrid_ranked
]

In [27]:
for item in hybrid_candidates:
    print(
        f"[{item['rrf_score']:.4f}] "
        f"{item['doc']['id']} | "
        f"{item['doc']['text']}"
    )

[0.0328] doc_3 | Error code ERR_CONNECTION_RESET occurs when the TCP connection is abruptly dropped by the server.
[0.0320] doc_1 | FastAPI is a modern, high-performance web framework for building APIs with Python.
[0.0161] doc_4 | PostgreSQL provides robust long-term user persistence and relational schema design.
[0.0159] doc_2 | LangGraph allows building stateful, multi-agent actor applications with graph topologies.


In [28]:
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4203.99it/s]


In [30]:
rerank_pairs = [
    [query, item["doc"]["text"]]
    for item in hybrid_candidates
]

rerank_pairs

[['How do I fix ERR_CONNECTION_RESET?',
  'Error code ERR_CONNECTION_RESET occurs when the TCP connection is abruptly dropped by the server.'],
 ['How do I fix ERR_CONNECTION_RESET?',
  'FastAPI is a modern, high-performance web framework for building APIs with Python.'],
 ['How do I fix ERR_CONNECTION_RESET?',
  'PostgreSQL provides robust long-term user persistence and relational schema design.'],
 ['How do I fix ERR_CONNECTION_RESET?',
  'LangGraph allows building stateful, multi-agent actor applications with graph topologies.']]

In [31]:
rerank_scores = reranker.predict(rerank_pairs)

rerank_scores

array([  4.962138, -11.434862, -11.3962  , -11.471806], dtype=float32)

In [32]:
reranked_results = []

for item, score in zip(hybrid_candidates, rerank_scores):
    reranked_results.append({
        "id": item["doc"]["id"],
        "text": item["doc"]["text"],
        "rerank_score": float(score),
    })

In [33]:
reranked_results = sorted(
    reranked_results,
    key=lambda x: x["rerank_score"],
    reverse=True,
)

In [34]:
for result in reranked_results:
    print(
        f"[{result['rerank_score']:.4f}] "
        f"{result['id']} | "
        f"{result['text']}"
    )

[4.9621] doc_3 | Error code ERR_CONNECTION_RESET occurs when the TCP connection is abruptly dropped by the server.
[-11.3962] doc_4 | PostgreSQL provides robust long-term user persistence and relational schema design.
[-11.4349] doc_1 | FastAPI is a modern, high-performance web framework for building APIs with Python.
[-11.4718] doc_2 | LangGraph allows building stateful, multi-agent actor applications with graph topologies.


### Full Hybrid Retrieval + Reranking Pipeline

**Query**
→ **BM25 Retrieval**
→ **Dense Retrieval**
→ **RRF Fusion**
→ **Hybrid Candidates**
→ **CrossEncoder Reranking**
→ **Final Ranked Results**

In [35]:
final_top_k = 2

final_results = reranked_results[:final_top_k]

In [36]:
for result in final_results:
    print(
        f"[{result['rerank_score']:.4f}] "
        f"{result['id']} | "
        f"{result['text']}"
    )

[4.9621] doc_3 | Error code ERR_CONNECTION_RESET occurs when the TCP connection is abruptly dropped by the server.
[-11.3962] doc_4 | PostgreSQL provides robust long-term user persistence and relational schema design.


In [37]:
context = "\n\n".join(
    result["text"]
    for result in final_results
)

print(context)

Error code ERR_CONNECTION_RESET occurs when the TCP connection is abruptly dropped by the server.

PostgreSQL provides robust long-term user persistence and relational schema design.


### Hybrid RAG Retrieval Pipeline

**User Query**

→ **Tokenize Query**

→ **BM25 Sparse Retrieval**

→ **Top BM25 Candidates**

+

**Query → SentenceTransformer → Query Embedding**

→ **Compare with Document Embeddings**

→ **Dense Similarity Scores**

→ **Top Dense Candidates**

↓

**Reciprocal Rank Fusion (RRF)**

→ **Hybrid Candidates**

→ **CrossEncoder Reranking**

→ **Final Top K Documents**

→ **Build Context**

→ **Context + User Query → LLM**